# Model app: Guanajuato orchestration

This notebook calls `run_model_app()` in `gradientdescentfarmers/app.py`. The `USER_PARAMETERS` dictionary below only overrides values from `resources/application.conf`; an empty dictionary runs the shared default. The app checks the configured latitude/longitude, reads the pinned Guanajuato GCS profile and SIAP files, prepares the documented weather analog and soil scenario, runs WOFOST, calibrates yield, selects price models, and returns the phase-level result. The notebook displays intermediate DataFrames and saves the one consolidated final CSV.

The pinned profile has weather for 2024–2025 only. The configured 2026–2027 weather is that history replayed as an analog, **not** measured future weather. The sample crop dates, 1-ha areas, and some hydraulic values are assumptions. Each phase runs independently. Until `source_builder` is connected, a different coordinate pair returns an explicit no-data message. The shared default contains two alternative two-phase maize schedules; change `rotation_options`, products, or water mode in `USER_PARAMETERS` when you have compatible inputs. Grain faba bean lacks 2024–2025 SIAP calibration overlap for this profile.

Set Google Application Default Credentials in the notebook kernel environment before running.


In [1]:
import sys
from pathlib import Path
import pandas as pd

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / "gradientdescentfarmers" / "model_builder").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook within the repository or install the package")
sys.path.insert(0, str(repo_root))

from gradientdescentfarmers.app import (
    NoPointDataError, PipelineConfig, run_model_app,
)
config_path = repo_root / "resources" / "application.conf"


In [2]:
# Override only the model inputs you want to change from application.conf.
# Other coordinates return NoPointDataError until source_builder is connected.
# For example: {"latitude": 20.4243195, "longitude": -101.1466639}
# You can also override rotation_options, selected_products, water_mode, etc.
USER_PARAMETERS = {}


In [3]:
file_config = PipelineConfig.from_file(config_path)
resolved = {**file_config.model, **USER_PARAMETERS}
print("Configured point:", resolved["latitude"], resolved["longitude"])
print("Pinned profile:", resolved["run_uri"])
print("Products:", resolved["selected_products"])
print("Water mode:", resolved["water_mode"])
print("Weather analog year shift:", resolved["weather_analog_years"])
for option in resolved["rotation_options"] or []:
    print("Rotation alternative:", option["rotation_id"], "phases:", len(option["phases"]))
print("The app resolves missing weather and soil inputs from the pinned profile.")


Configured point: 20.4243195 -101.1466639
Pinned profile: gs://data_dump_samples/farm_profiles/guanajuato_valle_demo/runs/67b66942681a4214
Products: ['maize']
Water mode: rainfed
Weather analog year shift: 2
Rotation alternative: maize_may_two_years phases: 2
Rotation alternative: maize_june_two_years phases: 2
The app resolves missing weather and soil inputs from the pinned profile.


In [4]:
try:
    result = run_model_app(USER_PARAMETERS, config_path=config_path)
except NoPointDataError as error:
    print(error)
    result = None

if result is not None:
    final_df = result.final_df
    preview = [
        "rotation_id", "rotation_step", "crop", "sowing_date", "harvest_date",
        "soil_moisture_end_m3_m3", "rainfall_total_mm",
        "soil_total_n_0_30cm_g_kg", "calibrated_t_ha", "estimated_tonnes",
        "estimated_price_mxn_t", "estimated_gross_value_mxn",
    ]
    print("Consolidated phase output:", final_df.shape)
    display(final_df[preview])
    print("Historical WOFOST terminal rows:")
    display(result.historical_daily_df.loc[
        result.historical_daily_df.is_final_day,
        ["crop", "year", "yield_t_ha", "water_mode"],
    ])
    print("SIAP yield calibration and held-out errors:")
    display(result.yield_calibration_df)
    display(result.yield_error_metrics_df)
    print("Price model selection and backtest sample:")
    display(result.price_leaderboard_df)
    display(result.price_backtest_df.head(10))
    print("Future daily WOFOST diagnostic shape:", result.future_daily_df.shape)
    print("Forecast horizon in months:", result.horizon_months)


Consolidated phase output: (4, 39)


,rotation_id,rotation_step,crop,sowing_date,harvest_date,soil_moisture_end_m3_m3,rainfall_total_mm,soil_total_n_0_30cm_g_kg,calibrated_t_ha,estimated_tonnes,estimated_price_mxn_t,estimated_gross_value_mxn
0,maize_june_two_years,1,maize,2026-06-01,2026-11-30,0.260827,784.62,1.784154,6.279721,6.279721,6040.966026,37935.579799
1,maize_june_two_years,2,maize,2027-06-01,2027-11-30,0.210725,660.99,1.784154,6.987473,6.987473,6595.891756,46088.614192
2,maize_may_two_years,1,maize,2026-05-15,2026-12-15,0.271960,791.17,1.784154,1.990274,1.990274,6040.966026,12023.180097
3,maize_may_two_years,2,maize,2027-05-15,2027-12-15,0.328707,732.08,1.784154,7.004326,7.004326,6595.891756,46199.777683


Historical WOFOST terminal rows:


,crop,year,yield_t_ha,water_mode
214,maize,2024,5.444421,rainfed
429,maize,2025,19.172086,rainfed


SIAP yield calibration and held-out errors:


,year,crop,simulated_t_ha,variety,water_mode,irrigation_system,irrigation_applied_mm,observed_t_ha,sown_ha,municipality,...,observation_scope,raw_error_t_ha,raw_absolute_error_t_ha,raw_absolute_percentage_error,calibration_factor,loyo_factor,calibrated_t_ha,loyo_t_ha,fit_error_t_ha,loyo_error_t_ha
0,2024,maize,5.444421,Grain_maize_202,rainfed,None,0,6.205779,17718.0,Valle de Santiago,...,municipal,-0.761359,0.761359,12.268544,0.36534,0.302882,1.989064,1.649017,-4.216716,-4.556763
1,2025,maize,19.172086,Grain_maize_202,rainfed,None,0,5.806878,17750.0,Valle de Santiago,...,municipal,13.365208,13.365208,230.161666,0.36534,1.139842,7.004326,21.853149,1.197448,16.046271


,prediction,n_years,mae_t_ha,rmse_t_ha,mean_bias_t_ha,mape_pct,crop,scope
0,loyo_t_ha,2,10.301517,11.795061,5.744754,174.879934,maize,municipal


Price model selection and backtest sample:


,model,mae_mxn_t,holdout_points,backtest_origins,crop,scope,forecast_horizon_years
0,ridge_lags,2105.619203,5,5,maize,municipal,2
1,last_value,2270.122695,5,5,maize,municipal,2
2,arima_110,2333.023354,5,5,maize,municipal,2
3,random_forest_lags,2346.038166,5,5,maize,municipal,2
4,holt_damped,2597.734797,5,5,maize,municipal,2


,model,origin_year,target_year,horizon_years,actual_mxn_t,predicted_mxn_t,absolute_error_mxn_t,crop
0,last_value,2019,2021,2,6122.973112,4018.167618,2104.805493,maize
1,last_value,2020,2022,2,7468.908295,4534.167369,2934.740926,maize
2,last_value,2021,2023,2,7849.567679,6122.973112,1726.594568,maize
3,last_value,2022,2024,2,5366.540917,7468.908295,2102.367378,maize
4,last_value,2023,2025,2,5367.462571,7849.567679,2482.105108,maize
5,holt_damped,2019,2021,2,6122.973112,3960.065648,2162.907463,maize
6,holt_damped,2020,2022,2,7468.908295,4080.065334,3388.842961,maize
7,holt_damped,2021,2023,2,7849.567679,6252.862211,1596.705469,maize
8,holt_damped,2022,2024,2,5366.540917,8053.547194,2687.006277,maize
9,holt_damped,2023,2025,2,5367.462571,8520.674385,3153.211814,maize


Future daily WOFOST diagnostic shape: (796, 83)
Forecast horizon in months: 24


In [5]:
SAVE_OUTPUT = True
if SAVE_OUTPUT and result is not None:
    output_path = result.save_final_csv(
        repo_root / "data" / "model_builder" / "app_profile_weather_analog_2026_2027_final.csv"
    )
    print("Saved one consolidated analog-scenario CSV:", output_path)


Saved one consolidated analog-scenario CSV: /Users/allisonlara/Desktop/NASA/GradientDescentFarmers/data/model_builder/app_profile_weather_analog_2026_2027_final.csv
